In [ ]:
from pathlib import Path
import copy
import sys
import json
from collections import defaultdict

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pypatchworkpp

from groundingdino.util.inference import load_model
from sam2.build_sam import build_sam2_video_predictor
from depth_anything_3.api import DepthAnything3
from depth_anything_3.utils.alignment import compute_sky_mask

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_window_contents,
    get_sweep_images_in_sample,
    get_lidar_pointcloud_in_sample,
    get_sample_data_2d_bboxes,
    filter_category_group_bboxes,
    CATEGORY_MAPPING_TO_UNIAD
)
from src.common.visualize.detection import plot_2d_boxes_on_image
from src.common.visualize.segmentation import (
    plot_instance_masks_on_image,
    plot_instance_mask
)
from src.common.visualize.depth import plot_depth_map
from src.common.visualize.pointcloud import plot_pointcloud
from src.common.visualize.colors import TABLEAU10_NAMES
from src.common.schemas import Box2D
from src.common.frame_ops import create_sliding_windows
from src.common.image_processing.segmentation import mask_morphology
from src.common.geometry.crop_resize import resize_mask_nearest
from src.common.geometry.depth import (
    depth_map_to_point_cloud_per_instance,
    transform_cam_to_ego
)
from src.common.geometry.pointcloud import (
    transform_lidar_to_ego,
    transform_ego_to_global
)

from src.grounding_dino.inference import predict_multi_labels
from src.sam2.inference import (
    init_frame_state,
    add_box_prompts,
    propagate_inference,
)
from src.sam2.utils import assign_continuous_tracking_ids
from src.depth_anything3.inference import get_metric_depth

# Resolve paths relative to this notebook directory
GROUNDINGDINO_CONFIG_PATH = ROOT / "GroundingDINO" / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
GROUNDINGDINO_WEIGHT_PATH = ROOT / "GroundingDINO" / "weights/groundingdino_swinb_cogcoor.pth"
SAM2_CONFIG_PATH = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM2_CHECKPOINT_PATH = ROOT / "sam2" / "checkpoints/sam2.1_hiera_large.pt"
DA3_MODEL_NAME = "DA3METRIC-LARGE"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Display parameters
NUM_SHOWN = 1
SHOWN_CHANNEL = "CAM_FRONT"

# Build the GroundingDINO model
groundingdino_model = load_model(str(GROUNDINGDINO_CONFIG_PATH), str(GROUNDINGDINO_WEIGHT_PATH), device=device)
# Build the SAM2 model and predictor
sam2_predictor = build_sam2_video_predictor(str(SAM2_CONFIG_PATH), str(SAM2_CHECKPOINT_PATH), device=device)
# Build the Depth-Anything-3 model
da3_model = DepthAnything3.from_pretrained(f"depth-anything/{DA3_MODEL_NAME}").to(device=device)
# Patchwork++
params = pypatchworkpp.Parameters()
PatchworkPLUSPLUS = pypatchworkpp.patchworkpp(params)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
CAMERA_CHANNELS = ["CAM_FRONT", "CAM_FRONT_RIGHT", "CAM_BACK_RIGHT", "CAM_BACK", "CAM_BACK_LEFT", "CAM_FRONT_LEFT"]
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

categories = {category["token"]: category for category in categories}
category_conversion = {k: v["category_name"] for k, v in CATEGORY_MAPPING_TO_UNIAD.items()}
category_names = list(dict.fromkeys(
    mapping["category_name"]
    for mapping in sorted(
        CATEGORY_MAPPING_TO_UNIAD.values(),
        key=lambda mapping: mapping["id"],
    )
))

# Get category groups
category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])

print(f"original_category_names: {[category['name'] for category in categories.values()]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Select the scenes and inference with GroundingDINO model
SCENE_NAME = "scene-0002"
MAX_SAMPLES = 10

# Define the frame interval for processing frames in the scene
FRAME_INTERVAL = 4
# Define the box thresholds for each category group
BOX_THRESHOLDS = {
    'vehicle': [0.25, 0.35, 0.45],
    'road_object': [0.20, 0.25, 0.30],
    'two_wheeler': [0.20, 0.30, 0.40],
    'pedestrian': [0.20, 0.30, 0.40],
}

# Get the scene contents for the selected scene
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
scene_contents = get_scene_contents(scene["token"], samples_all, 
                                    sample_data_all, ego_poses_all, calibrated_sensors_all,
                                    get_non_key_frames=True,
                                    sample_annotations_all=sample_annotations_all,
                                    instances_all=instances_all,
                                    max_samples=MAX_SAMPLES)
samples = scene_contents["samples"]
sample_data = scene_contents["sample_data"]
keyframe_sample_data = {k: v for k, v in sample_data.items() if v["is_key_frame"]}
ego_poses = scene_contents["ego_poses"]
calibrated_sensors = scene_contents["calibrated_sensors"]
sample_annotations = scene_contents["sample_annotations"]
instances = scene_contents["instances"]
# Create ground truth track_ids from instance tokens
gt_track_ids = {inst_token: i for i, inst_token in enumerate(instances.keys())}

# Build frame indices with interval and always include the last frame index
proc_sample_indices = list(range(0, len(samples), FRAME_INTERVAL))
if proc_sample_indices and proc_sample_indices[-1] != len(samples) - 1:
    proc_sample_indices.append(len(samples) - 1)

det2d_results = {}
# Frame loop
for i, sample_index in enumerate(proc_sample_indices):
    det2d_results[sample_index] = {}
    # Camera channel loop
    for camera_channel in CAMERA_CHANNELS:
        print(f"Processing frame {sample_index}/{len(samples) - 1} for camera channel {camera_channel}")
        det2d_results[sample_index][camera_channel] = {}
        # Get the KEYFRAME contents of the current sample and the specified camera channel
        sample_contents_cam = get_sample_contents(sample_index, samples, keyframe_sample_data, ego_poses, calibrated_sensors,
                                                  sensor_token=sensor_lookup[camera_channel])
        sample_data_cam = list(sample_contents_cam["sample_data"].values())[0]
        # Load the image for inference
        image_path = NUSCENES_ROOT / sample_data_cam["filename"]
        image = Image.open(image_path)

        # Create canvas for plotting the results
        if i < NUM_SHOWN and camera_channel == SHOWN_CHANNEL:
            num_cols = 1 + len(list(BOX_THRESHOLDS.values())[0])  # +1 for the ground truth boxes
            num_rows = len(category_groups)  # Number of category groups
            fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6 * num_cols, 4 * num_rows))
            # Get the ground truth bounding boxes in the camera frame for the sample data
            gt_boxes_3d, gt_boxes_2d = get_sample_data_2d_bboxes(
                sample_data=sample_data_cam,
                sample_annotations=sample_annotations,
                instances=instances,
                categories=categories,
                calibrated_sensors=calibrated_sensors,
                ego_poses=ego_poses,
                category_conversion=category_conversion,
                track_ids=gt_track_ids
            )
        
        # Category group loop
        for category_group_index, category_group in enumerate(category_groups):
            category_names_in_group = set([v['category_name'] for v in CATEGORY_MAPPING_TO_UNIAD.values() 
                                           if v['category_group'] == category_group])

            # Plot the ground truth boxes for the category group
            if i < NUM_SHOWN and camera_channel == SHOWN_CHANNEL:
                category_gt_boxes_3d, category_gt_boxes_2d = filter_category_group_bboxes(
                    boxes_3d=gt_boxes_3d,
                    boxes_2d=gt_boxes_2d,
                    category_group=category_group,
                    category_mapping=CATEGORY_MAPPING_TO_UNIAD
                )
                print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {len(category_gt_boxes_2d)}")
                category_colors = {category_name: TABLEAU10_NAMES[cat_idx % len(TABLEAU10_NAMES)]
                                for cat_idx, category_name in enumerate(category_names_in_group)}
                plot_2d_boxes_on_image(image, category_gt_boxes_2d,
                                    ax=axes[category_group_index, 0],
                                    color=category_colors,
                                    title=f"category_group:{category_group}, GT boxes")
            
            # Iterate over different box thresholds
            for box_threshold_index, box_threshold in enumerate(BOX_THRESHOLDS[category_group]):
                # Infer the image with GroundingDINO model
                predicted_boxes, caption = predict_multi_labels(
                    model=groundingdino_model,
                    image=image,
                    labels=category_names_in_group,
                    box_threshold=box_threshold,
                )
                # convert box coordinates from normalized to pixel coordinates
                for box in predicted_boxes:
                    box.xyxy = box.xyxy * np.array([image.width, image.height, image.width, image.height])
                
                # Store the results of the second threshold for later use
                if box_threshold_index == 1:
                    print(f"Group: {category_group}. Detected {len(predicted_boxes)} boxes above the threshold of {box_threshold}")
                    det2d_results[sample_index][camera_channel][category_group] = {
                        "box_threshold": box_threshold,
                        "predicted_boxes": predicted_boxes,
                        "caption": caption
                    }
                
                # Plot
                if i < NUM_SHOWN and camera_channel == SHOWN_CHANNEL:
                    plot_2d_boxes_on_image(image, predicted_boxes,
                                        ax=axes[category_group_index, box_threshold_index + 1],
                                        color=category_colors,
                                        title=f"{category_group}, box_threshold:{box_threshold}")
        plt.show()


In [ ]:
# SAM2 instance segmentation
TRACKING_NUM_SWEEPS = 2  # Number of frames per keyframe to use for tracking (including the keyframe itself)
TRACKING_IOU_METHOD = "box"  # "box" or "mask"
TRACKING_IOU_THRESHOLD = 0.5  # IoU threshold for determining if the object ids are the same across frames
TRACKING_IOU_MATCH_LABEL = True  # Whether to match the category labels when determining if the object ids are the same across frames

color_dict = {i: TABLEAU10_NAMES[i] for i in range(10)}

instance_tracking_results = {}
# Camera channel loop
for camera_channel in CAMERA_CHANNELS:
    max_track_id = -1  # Initialize max_track_id for each camera channel
    instance_tracking_results[camera_channel] = {}
    # Frame loop
    for i, sample_index in enumerate(proc_sample_indices):
        print(f"Processing frame {sample_index}/{len(samples) - 1} for camera channel {camera_channel}")
        next_sample_index = proc_sample_indices[i + 1] if i < len(proc_sample_indices) - 1 else sample_index
        
        # Load the images
        images = []
        timestamps = []
        keyframe_indices = []
        for s_index in range(sample_index, next_sample_index + 1):
            # Non-keyframe in the first sample is not used for inference
            if len(images) == 0:
                keyframe_sample_contents = get_sample_contents(s_index, samples, keyframe_sample_data, ego_poses, calibrated_sensors,
                                                               sensor_token=sensor_lookup[camera_channel])
                keyframe_in_sample = list(keyframe_sample_contents["sample_data"].values())[0]
                images_in_sample = [Image.open(NUSCENES_ROOT / keyframe_in_sample["filename"])]
                timestamps_in_sample = [keyframe_in_sample["timestamp"]]
            # Use non-keyframe images in the subsequent samples for inference
            else:
                images_in_sample, timestamps_in_sample = get_sweep_images_in_sample(
                    sample_index=s_index,
                    samples_in_scene=samples,
                    sample_data_in_scene=sample_data,
                    ego_poses_in_scene=ego_poses,
                    calibrated_sensors_in_scene=calibrated_sensors,
                    camera_sensor_token=sensor_lookup[camera_channel],
                    nuscenes_root=NUSCENES_ROOT,
                    nsweeps=TRACKING_NUM_SWEEPS
                )
            images.extend(images_in_sample)
            timestamps.extend(timestamps_in_sample)
            keyframe_indices.append(len(images) - 1)

        # Initialize the inference state for SAM2 with the loaded images
        inference_state = init_frame_state(sam2_predictor, images)

        # Get the GroundingDINO result
        frame_det2d_boxes = [box for result in det2d_results[sample_index][camera_channel].values()
                             for box in result["predicted_boxes"]]
        # Temporary track_ids (Re-assign track_ids based on IoU with previous frame later)
        for i_box in range(len(frame_det2d_boxes)):
            frame_det2d_boxes[i_box].track_id = i_box
        tmp_track_id_to_label = {box.track_id: box.label for box in frame_det2d_boxes}

        # Reset and Add the box prompts
        sam2_predictor.reset_state(inference_state)
        predicted_instances = add_box_prompts(
            predictor=sam2_predictor,
            inference_state=inference_state,
            frame_idx=0,
            box_prompts=frame_det2d_boxes,
        )
        # propagate the tracking inference
        full_result_instances = propagate_inference(sam2_predictor, inference_state)

        # Identify the track_id based on IoU with propagated instances from the previous frame
        if i == 0:  # first sample in the sequence
            track_id_mapping = {i: i for i in range(len(frame_det2d_boxes))}
            max_track_id = max(track_id_mapping.values())
        else:
            idx_to_track_id, max_track_id = assign_continuous_tracking_ids(
                current_predicted_instances=predicted_instances,
                prev_propagated_instances=prev_propagated_instances,
                max_track_id=max_track_id,
                iou_method=TRACKING_IOU_METHOD,
                iou_threshold=TRACKING_IOU_THRESHOLD,
                match_label=TRACKING_IOU_MATCH_LABEL
            )
            track_id_mapping = {instance.box.track_id: idx_to_track_id[predicted_inst_idx] for predicted_inst_idx, instance in enumerate(predicted_instances)}

        # Update the labels and track_ids
        for propagate_frame_idx, frame_result_instances in full_result_instances.items():
            for tmp_track_id, instance in frame_result_instances.items():
                instance.box.label = tmp_track_id_to_label[tmp_track_id]
                instance.box.track_id = track_id_mapping[tmp_track_id]
        for instance in predicted_instances:
            instance.box.track_id = track_id_mapping[instance.box.track_id]

        # Filter the results to only include the keyframes
        result_instances = {k: v for i_sweep, (k, v) in enumerate(full_result_instances.items()) if i_sweep in keyframe_indices}

        # Show the current predicted instances and the previous propagated instances for comparison
        color_map = {track_id: color_dict[track_id % 10] for track_id in range(max_track_id + 1)}
        if i > 0 and i < NUM_SHOWN and camera_channel == SHOWN_CHANNEL:
            fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))
            current_predicted_boxes = [Box2D(xyxy=np.array(instance.mask_region), label=instance.box.label, track_id=instance.box.track_id)
                                       for instance in predicted_instances]
            plot_instance_masks_on_image(
                instances=prev_propagated_instances,
                image=images[0],
                ax=axes[0],
                title=f"Masks propagated from sample {proc_sample_indices[i - 1]} to sample {sample_index}.\nBoxes are reference from current prediction",
                color=color_map,
                color_attr="track_id",
                line_width=0.5,
                text_shown_attr="track_id"
            )
            for predicted_inst_idx, instance in enumerate(predicted_instances):  # Show the boxes of the current predicted instances on the left subplot for comparison
                x1, y1, x2, y2 = instance.box.xyxy
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor=color_map[instance.box.track_id], facecolor='none', linewidth=0.5)
                axes[0].add_patch(rect)
            plot_instance_masks_on_image(
                instances=predicted_instances,
                image=images[0],
                prompt_boxes=current_predicted_boxes,
                ax=axes[1],
                title=f"Masks predicted in sample {sample_index} with box prompts",
                color=color_map,
                prompt_box_color=color_map,
                color_attr="track_id",
                line_width=0.5,
                text_shown_attr="track_id"
            )
            plt.show()
        

        # Store the results for the current frame
        for propagate_frame_idx, frame_result_instances in result_instances.items():
            if propagate_frame_idx == max(result_instances.keys()): # The last frame in the window doesn't used as a result, but used for the track_id update
                prev_propagated_instances = list(frame_result_instances.values())
            if propagate_frame_idx in keyframe_indices:  # Only store the results for the keyframes
                instance_tracking_results[camera_channel][sample_index + keyframe_indices.index(propagate_frame_idx)] = list(frame_result_instances.values())
        

        if i < NUM_SHOWN and camera_channel == SHOWN_CHANNEL:
            # Plot the propagated instance masks for the frames in the window
            num_cols = TRACKING_NUM_SWEEPS
            num_rows = FRAME_INTERVAL + 1
            fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(8 * num_cols, 5 * num_rows))
            sample_count = 0
            frame_count = num_cols - 1
            for propagate_frame_idx, frame_result_instances in full_result_instances.items():
                plot_instance_masks_on_image(
                    instances=list(frame_result_instances.values()),
                    image=images[propagate_frame_idx],
                    prompt_boxes=frame_det2d_boxes if propagate_frame_idx == 0 else None,
                    image_alpha=0.6,
                    mask_alpha=0.7,
                    ax=axes[sample_count, frame_count],
                    title=f"Sample {sample_count}" + (" (keyframe)" if propagate_frame_idx in keyframe_indices else " (non-keyframe)") + (" with box prompts" if propagate_frame_idx == 0 else " propagated masks"),
                    color=color_map,
                    prompt_box_color=color_map,
                    color_attr="track_id",
                    line_width=0.5,
                    text_shown_attr="track_id"
                )
                if propagate_frame_idx in keyframe_indices:
                    sample_count += 1
                    frame_count = 0
                else:
                    frame_count += 1
            plt.show()

In [ ]:
# Depth-Anything-3 Inference
# Pose-conditioned model parameters
WINDOW_SIZE = 9  # Number of frames in the sliding window
STRIDE = 5  # Stride for sliding window

use_pose_cond = DA3_MODEL_NAME in ["DA3NESTED-GIANT-LARGE", "DA3NESTED-GIANT-LARGE-1.1"]

depth_est_results = {}
# Camera channel loop
for camera_channel in CAMERA_CHANNELS:
    max_track_id = -1  # Initialize max_track_id for each camera channel
    depth_est_results[camera_channel] = {}

    ###### Pose-conditioned depth estimation ######
    if use_pose_cond:
        # Create a sliding windows
        window_ranges, used_ranges = create_sliding_windows(len(samples), window_size=WINDOW_SIZE, stride=STRIDE)
        for window_count, ((i_start, i_end), (u_start, u_end)) in enumerate(zip(window_ranges, used_ranges)):
            print(f"Processing frame {i_start}/{len(samples) - 1} for camera channel {camera_channel}")
            # Get the KEYFRAME contents of the current sliding window
            window_contents_cam = get_sample_window_contents(i_start, i_end-1, samples, keyframe_sample_data, ego_poses, calibrated_sensors,
                                                            sensor_token=sensor_lookup[camera_channel])
            
            window_samples = window_contents_cam["samples"]
            window_sample_data_cam = window_contents_cam["sample_data"]
            window_ego_poses_cam = window_contents_cam["ego_poses"]
            window_calibrated_sensors_cam = window_contents_cam["calibrated_sensors"]

            # Get the images and camera intrinsics for the sliding window
            images = [Image.open(NUSCENES_ROOT / sd["filename"]) for sd in window_sample_data_cam.values()]
            intrinsics = []
            extrinsics = []

            # Calculate the extrinsics for the sliding window
            for sd in window_sample_data_cam.values():
                cs = window_calibrated_sensors_cam[sd["calibrated_sensor_token"]]
                intrinsics.append(cs["camera_intrinsic"])
                ep = window_ego_poses_cam[sd["ego_pose_token"]]
                camera_to_ego = make_transform(quaternion=cs["rotation"], translation=cs["translation"])
                ego_to_global = make_transform(quaternion=ep["rotation"], translation=ep["translation"])
                camera_to_global = ego_to_global @ camera_to_ego
                global_to_camera = invert_transform(camera_to_global)
                extrinsics.append(global_to_camera)
            
            # Stack intrinsics and extrinsics for the sliding window
            intrinsics = np.stack(intrinsics, axis=0).astype(np.float32)
            extrinsics = np.stack(extrinsics, axis=0).astype(np.float32)

            # Inference depth using DepthAnything3 with pose conditioning
            prediction = da3_model.inference(images, extrinsics=extrinsics, intrinsics=intrinsics)
            metric_depths, scaled_intrinsics = get_metric_depth(prediction, model_name=DA3_MODEL_NAME)
        
        # Store the depth estimation results for the used frames in the sliding window
        for used_idx in range(u_start, u_end):
            used_sample_data = list(window_sample_data_cam.values())[used_idx]
            depth_est_results[camera_channel][i_start + used_idx] = {
                "metric_depth": metric_depths[used_idx],
                "scaled_intrinsics": scaled_intrinsics[used_idx],
                "non_sky_mask": compute_sky_mask(prediction.sky[used_idx]) if prediction.sky is not None else None,
                "camera_translation": window_calibrated_sensors_cam[used_sample_data["calibrated_sensor_token"]]["translation"],
                "camera_quaternion": window_calibrated_sensors_cam[used_sample_data["calibrated_sensor_token"]]["rotation"],
                "ego_translation": window_ego_poses_cam[used_sample_data["ego_pose_token"]]["translation"],
                "ego_quaternion": window_ego_poses_cam[used_sample_data["ego_pose_token"]]["rotation"],
            }
    
    ###### Single frame depth estimation ######
    else:
        # Sample loop
        for sample_index in range(len(samples)):
            print(f"Processing sample {sample_index}/{len(samples) - 1} for camera channel {camera_channel}")
            # Get the KEYFRAME contents of the current sample
            sample_contents_cam = get_sample_contents(sample_index, samples, keyframe_sample_data, ego_poses, calibrated_sensors,
                                                      sensor_token=sensor_lookup[camera_channel])
            sample_data_cam = list(sample_contents_cam["sample_data"].values())[0]
            calibrated_sensor_cam = list(sample_contents_cam["calibrated_sensors"].values())[0]
            ego_pose_cam = list(sample_contents_cam["ego_poses"].values())[0]

            # Inference depth using DepthAnything3 (without pose conditioning)
            image = Image.open(NUSCENES_ROOT / sample_data_cam["filename"])
            prediction = da3_model.inference([image])
            metric_depths, scaled_intrinsics = get_metric_depth(prediction,
                                                                model_name=DA3_MODEL_NAME,
                                                                camera_intrinsics=[calibrated_sensor_cam["camera_intrinsic"]],
                                                                original_image_width=image.width,
                                                                original_image_height=image.height)

            # Store the depth estimation results
            depth_est_results[camera_channel][sample_index] = {
                "metric_depth": metric_depths[0],
                "scaled_intrinsics": scaled_intrinsics[0],
                "non_sky_mask": compute_sky_mask(prediction.sky[0]) if prediction.sky is not None else None,
                "original_image_width": image.width,
                "original_image_height": image.height,
                "depth_image_width": metric_depths[0].shape[1],
                "depth_image_height": metric_depths[0].shape[0],
                "camera_translation": calibrated_sensor_cam["translation"],
                "camera_quaternion": calibrated_sensor_cam["rotation"],
                "ego_translation": ego_pose_cam["translation"],
                "ego_quaternion": ego_pose_cam["rotation"],
            }

In [ ]:
# Project the depth maps and LiDAR point clouds to the instance masks
NUM_LIDAR_SWEEPS = 3  # Number of LiDAR sweeps to use for point cloud projection
CLOSING_KERNEL_SIZES = [3, -3]  # Kernel sizes for morphological closing operation. Positive size represents dilation, whereas negative size represents erosion.
RATIO_MORPHOLOGY = False  # Whether to apply ratio-based morphology. If True, the applied kernel sizes are computed by multiplying `kernel_sizes` by the average of the mask bounding box height and width.

category_color_dict = {category_name: TABLEAU10_NAMES[cat_idx % len(TABLEAU10_NAMES)]
                       for cat_idx, category_name in enumerate(category_names)}


NUM_SHOWN=3


pointclouds_per_instance = {}
# Frame loop
for sample_index in range(len(samples)):
    pointclouds_per_instance[sample_index] = {}

    # Read the LiDAR point cloud
    lidar_data = get_lidar_pointcloud_in_sample(
        sample_index=sample_index,
        samples_in_scene=samples,
        sample_data_in_scene=sample_data,
        ego_poses_in_scene=ego_poses,
        calibrated_sensors_in_scene=calibrated_sensors,
        lidar_sensor_token=sensor_lookup["LIDAR_TOP"],
        nuscenes_root=NUSCENES_ROOT,
        nsweeps=NUM_LIDAR_SWEEPS,
        stack_result=False,
    )
    lidar_translation=lidar_data[-1]["lidar_translation"]
    lidar_quaternion=lidar_data[-1]["lidar_quaternion"]
    ego_translation=lidar_data[-1]["ego_translation"]
    ego_quaternion=lidar_data[-1]["ego_quaternion"]
    # Remove the ground points by Patchwork++
    nonground = []
    for sweep_lidar in lidar_data:
        pointcloud = np.hstack((sweep_lidar["points"], sweep_lidar["intensity"].reshape(-1, 1)))  # Combine points and intensity into a single array
        PatchworkPLUSPLUS.estimateGround(pointcloud)
        sweep_ground = PatchworkPLUSPLUS.getGround()
        sweep_nonground = PatchworkPLUSPLUS.getNonground()
        nonground.append(sweep_nonground)
    lidar_points = np.vstack(nonground)

    # Create canvas for plotting the depth maps and pseudo-LiDAR point clouds for the first few frames
    if sample_index < NUM_SHOWN:
        num_cols = 3
        num_rows = len(CAMERA_CHANNELS)
        fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(8 * num_cols, 5 * num_rows))
        point_colors = defaultdict(dict)

    # Camera channel loop
    for i_cam, camera_channel in enumerate(CAMERA_CHANNELS):
        ###### Depth map projection ######
        pointclouds_per_instance[sample_index][camera_channel] = {}

        depth_est_result = depth_est_results[camera_channel][sample_index]
        depth_image = depth_est_result["metric_depth"]
        # Resize the instance masks to the depth image size
        instance_masks = instance_tracking_results[camera_channel][sample_index]
        depth_instance_masks = [instance.convert_to_original_coordinates(
                                    original_width=depth_est_result["depth_image_width"],
                                    original_height=depth_est_result["depth_image_height"],
                                    input_width=depth_est_result["original_image_width"],
                                    input_height=depth_est_result["original_image_height"]
                                )
                                for instance in instance_masks]
        # Instance mask closing operation to fill holes and remove noise
        closed_instance_masks = [mask_morphology(mask, kernel_sizes=CLOSING_KERNEL_SIZES, ratio_morphology=RATIO_MORPHOLOGY)
                                for mask in depth_instance_masks]
        
        # Get the pseudo-LiDAR point clouds from the depth image and instance masks
        pseudo_points_per_instance, colors = depth_map_to_point_cloud_per_instance(
            metric_depth=depth_image,
            camera_intrinsics=depth_est_result["scaled_intrinsics"],
            instances=closed_instance_masks,
            common_mask=depth_est_result["non_sky_mask"],
            color=category_color_dict,
            color_attr="label"
        )
        for instance, pseudo_points in zip(closed_instance_masks, pseudo_points_per_instance):
            pseudo_points_ego = transform_cam_to_ego(
                pseudo_points,
                camera_translation=depth_est_result["camera_translation"],
                camera_quaternion=depth_est_result["camera_quaternion"]
            )
            pseudo_points_global = transform_ego_to_global(
                pseudo_points_ego,
                ego_translation=depth_est_result["ego_translation"],
                ego_quaternion=depth_est_result["ego_quaternion"]
            )
            pointclouds_per_instance[sample_index][camera_channel][instance.box.track_id] = {}
            pointclouds_per_instance[sample_index][camera_channel][instance.box.track_id]["pseudo_pointcloud"] = pseudo_points_global


        ###### LiDAR point projection ######



        # Visualization        
        if sample_index < NUM_SHOWN:
            # Plot the depth map
            plot_depth_map(depth_image, platform="matplotlib", ax=axes[i_cam, 0], title=f"Depth map, {camera_channel}, sample {sample_index}")
            plot_depth_map(depth_image, platform="matplotlib", ax=axes[i_cam, 1], title=f"Depth map with instance masks")
            plot_depth_map(depth_image, platform="matplotlib", ax=axes[i_cam, 2], title=f"Depth map with CLOSED instance masks")
            for instance_idx in range(len(closed_instance_masks)):
                plot_instance_mask(depth_instance_masks[instance_idx],
                                   image_height=depth_image.shape[0], image_width=depth_image.shape[1],
                                   ax=axes[i_cam, 1], color=category_color_dict[depth_instance_masks[instance_idx].box.label],
                                   alpha=0.6)
                plot_instance_mask(closed_instance_masks[instance_idx],
                                   image_height=depth_image.shape[0], image_width=depth_image.shape[1],
                                   ax=axes[i_cam, 2], color=category_color_dict[closed_instance_masks[instance_idx].box.label],
                                   alpha=0.6)
            
            # Store the pseudo-LiDAR point colors for plotting
            for instance, colors in zip(closed_instance_masks, colors):
                point_colors[camera_channel][instance.box.track_id] = colors
                
    if sample_index < NUM_SHOWN:
        # Transform the points from the LiDAR frame to the global frame
        lidar_points_ego = transform_lidar_to_ego(lidar_points, 
                                                  lidar_translation=lidar_translation,
                                                  lidar_quaternion=lidar_quaternion)
        lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                        ego_translation=ego_translation,
                                                        ego_quaternion=ego_quaternion)
        # Combine the LiDAR points and pseudo-LiDAR points for plotting
        combined_points = [lidar_points_global]
        combined_colors = [np.tile(np.array([[0.8, 0.8, 0.8]]), (lidar_points_global.shape[0], 1))]  # Gray color for LiDAR points
        for camera_channel in pointclouds_per_instance[sample_index]:
            for instance_id in pointclouds_per_instance[sample_index][camera_channel]:
                combined_points.append(pointclouds_per_instance[sample_index][camera_channel][instance_id]["pseudo_pointcloud"])
                combined_colors.append(point_colors[camera_channel][instance_id])
        combined_points = np.concatenate(combined_points, axis=0)
        combined_colors = np.concatenate(combined_colors, axis=0)
        # Plot the pseudo-LiDAR point clouds
        plot_fig = plot_pointcloud(combined_points,
                                   colors=combined_colors,
                                   axis_translation=depth_est_result["ego_translation"],
                                   axis_quaternion=depth_est_result["ego_quaternion"],
                                   fig_width=960, fig_height=720,
                                   point_size=0.5,
                                   title=f"Sample {sample_index}, LiDAR + Pseudo-LiDAR point clouds"
                                   )
        plot_fig.show()
        # Show the depth map and instance masks for each camera channel
        plt.tight_layout()
        plt.show()

        ###### LiDAR projection ######

In [ ]:
# 3D box fitting

In [ ]:
# SAM2 instance segmentation with tracklet matching tracking across frames
# IoU calculation method to determine if the object ids are the same across frames.
TRACKING_IOU_METHOD = "box"  # "box" or "mask"
TRACKING_IOU_THRESHOLD = 0.5  # IoU threshold for determining if the object ids are the same across frames
TRACKING_IOU_MATCH_LABEL = True  # Whether to match the category labels when determining if the object ids are the same across frames

color_dict = {i: TABLEAU10_NAMES[i] for i in range(10)}

###### Instance tracking across frames ######
window_tracking_results = {}

# Frame loop
for i, sample_index in enumerate(proc_sample_indices):
    if i >= len(proc_sample_indices) - 1:
        break  # Skip the last frame since it doesn't have a next frame for propagation
    window_tracking_results[sample_index] = {}

    # Camera channel loop
    for camera_channel in CAMERA_CHANNELS:
        print(f"Processing frame {sample_index}/{len(samples) - 1} for camera channel {camera_channel}")
        window_tracking_results[sample_index][camera_channel] = {}

        # Get the contents between the current and the next sample and the specified camera channel
        next_sample_index = proc_sample_indices[i + 1]
        num_propagated_frames = next_sample_index - sample_index + 1
        window_contents_cam = get_sample_window_contents(sample_index, next_sample_index, 
                                                         samples, sample_data, ego_poses, calibrated_sensors,
                                                         sensor_token=sensor_lookup[camera_channel])
        window_sample_data_cam = window_contents_cam["sample_data"]
        # Load the images and set them to inference_state
        images = [Image.open(NUSCENES_ROOT / sample_data["filename"]) for sample_data in window_sample_data_cam.values()]
        inference_state = init_frame_state(sam2_predictor, images)

        ###### Forward propagation ######
        # Get the GroundingDINO result
        forward_det2d_boxes = [box for result in det2d_results[sample_index][camera_channel].values()
                               for box in result["predicted_boxes"]]
        # Temporary track_ids (Re-assign track_ids based on IoU with previous frame later)
        for i_box in range(len(forward_det2d_boxes)):
            forward_det2d_boxes[i_box].track_id = i_box
        forward_tmp_track_id_to_label = {box.track_id: box.label for box in forward_det2d_boxes}
        # Reset and Add the box prompts
        sam2_predictor.reset_state(inference_state)
        forward_predicted_instances = add_box_prompts(
            predictor=sam2_predictor,
            inference_state=inference_state,
            frame_idx=0,
            box_prompts=forward_det2d_boxes,
        )
        # propagate the tracking inference
        forward_instances = propagate_inference(sam2_predictor, inference_state)
        forward_instances = {k: forward_instances[k] for k in sorted(forward_instances.keys())}  # Sort by frame index

        ###### Backward propagation ######
        # Get the GroundingDINO result
        backward_det2d_boxes = [box for result in det2d_results[next_sample_index][camera_channel].values()
                               for box in result["predicted_boxes"]]
        # Temporary track_ids (Re-assign track_ids based on IoU with previous frame later)
        for i_box in range(len(backward_det2d_boxes)):
            backward_det2d_boxes[i_box].track_id = i_box
        backward_tmp_track_id_to_label = {box.track_id: box.label for box in backward_det2d_boxes}
        # Reset and Add the box prompts
        sam2_predictor.reset_state(inference_state)
        backward_predicted_instances = add_box_prompts(
            predictor=sam2_predictor,
            inference_state=inference_state,
            frame_idx=num_propagated_frames - 1,
            box_prompts=backward_det2d_boxes,
        )
        # propagate the tracking inference
        backward_instances = propagate_inference(sam2_predictor, inference_state,
                                                 start_frame_idx=num_propagated_frames - 1,
                                                 reverse=True)
        backward_instances = {k: backward_instances[k] for k in sorted(backward_instances.keys())}  # Sort by frame index

        # Store the results for the current frame
        for propagate_frame_idx, frame_result_instances in result_instances.items():
            window_tracking_results[sample_index][camera_channel]['forward'] = forward_instances
            window_tracking_results[sample_index][camera_channel]['backward'] = backward_instances

        # Show the forward and backward propagated instances for comparison
        if i < NUM_SHOWN and camera_channel == SHOWN_CHANNEL:
            max_track_id = max(len(forward_det2d_boxes), len(backward_det2d_boxes)) - 1
            color_map = {track_id: color_dict[track_id % 10] for track_id in range(max_track_id + 1)}
            num_cols = 2
            num_rows = FRAME_INTERVAL + 1
            fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(8 * num_cols, 5 * num_rows))
            for propagate_frame_idx in range(len(images)):
                # Plot the forward propagated instance masks for the frames in the window
                plot_instance_masks_on_image(
                    instances=list(forward_instances[propagate_frame_idx].values()),
                    image=images[propagate_frame_idx],
                    prompt_boxes=forward_det2d_boxes if propagate_frame_idx == 0 else None,
                    image_alpha=0.6,
                    mask_alpha=0.7,
                    ax=axes[propagate_frame_idx, 0],
                    title=f"Forward propagated masks from frame {sample_index} to frame {next_sample_index}.",
                    color=color_map,
                    prompt_box_color=color_map,
                    color_attr="track_id",
                    line_width=0.5,
                    text_shown_attr="track_id"
                )
                # Plot the backward propagated instance masks for the frames in the window
                plot_instance_masks_on_image(
                    instances=list(backward_instances[propagate_frame_idx].values()),
                    image=images[propagate_frame_idx],
                    prompt_boxes=backward_det2d_boxes if propagate_frame_idx == len(backward_instances) - 1 else None,
                    image_alpha=0.6,
                    mask_alpha=0.7,
                    ax=axes[propagate_frame_idx, 1],
                    title=f"Backward propagated masks from frame {next_sample_index} to frame {sample_index}.",
                    color=color_map,
                    prompt_box_color=color_map,
                    color_attr="track_id",
                    line_width=0.5,
                    text_shown_attr="track_id"
                )
            plt.show()

# TODO: Conduct tracklet matching between forward_instances and backward_instances and update the track_ids accordingly. This part is not implemented in the provided code.


